# 06 — Manual error review and student findings

**Viraat Chaudhary · DATA 266 Lab 1 · Team 15 · Yelp Polarity**

Finish notebook 05 first. CPU Colab is sufficient. The review model was selected by validation before test. Review exactly 20 unique errors: five confident FP, five confident FN, five near-threshold, five slice-specific. Ranked confidence/threshold definitions are recorded; read actual confidence values rather than assuming a fixed cutoff.

Run top-to-bottom. Save the executed notebook using **File → Download → Download .ipynb**. Use the exact filename provided. Outputs persist in your own Drive run folder. Source code is assistant-assisted; student explanations and analysis must be your own.


In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import drive, files

# Set these once; keep the same RUN_NAME for notebooks 02–07 and resumed sessions.
DRIVE_BASE_RELATIVE = 'DATA266_Lab1_Task2/Viraat_Chaudhary'
RUN_NAME = 'task2_models_rnn_lstm_tcn_v1'
PACKAGE_ZIP_NAME = 'Viraat_Task2_Run_Package.zip'
PACKAGE_ID = 'viraat_yelp_rnn_lstm_tcn_v1'

drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive') / DRIVE_BASE_RELATIVE
ROOT = DRIVE_BASE / RUN_NAME / 'task2_sentiment/Viraat_Chaudhary'
CACHE_ROOT = Path('/content/viraat_task2_data')
if not ROOT.exists():
    archive = DRIVE_BASE / PACKAGE_ZIP_NAME
    if not archive.exists():
        print('Upload Viraat_Task2_Run_Package.zip (the code package, not a dataset ZIP).')
        uploaded = files.upload()
        if PACKAGE_ZIP_NAME not in uploaded:
            raise FileNotFoundError('Select the exact package ZIP filename.')
        archive = Path('/content') / PACKAGE_ZIP_NAME
        archive.write_bytes(uploaded[PACKAGE_ZIP_NAME])
    extraction = Path('/content/viraat_task2_package_extract')
    extraction.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            if not (extraction / info.filename).resolve().is_relative_to(extraction.resolve()):
                raise ValueError('Invalid package ZIP path.')
        if bundle.testzip() is not None:
            raise ValueError('Package ZIP CRC failed.')
        bundle.extractall(extraction)
    manifest = json.loads((extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_text())
    if manifest['package_id'] != PACKAGE_ID:
        raise ValueError('Incorrect package identity.')
    for item in manifest['files']:
        path = extraction / item['path']
        if hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
            raise ValueError('Package integrity failed: ' + item['path'])
    source = extraction / 'task2_sentiment/Viraat_Chaudhary'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, ROOT)
    (ROOT / 'package_installation.json').write_text(json.dumps({'package_id': PACKAGE_ID,
                                                              'package_manifest_sha256': hashlib.sha256(
        (extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_bytes()).hexdigest()}, indent=2) + '\n')
else:
    installed = json.loads((ROOT / 'package_installation.json').read_text())
    if installed['package_id'] != PACKAGE_ID:
        raise ValueError('This RUN_NAME belongs to a different package.')

required = {'numpy': 'numpy==2.1.3', 'scipy': 'scipy>=1.14,<2',
            'sklearn': 'scikit-learn==1.6.1', 'nltk': 'nltk==3.9.1',
            'datasets': 'datasets==4.8.5', 'matplotlib': 'matplotlib==3.10.0',
            'pandas': 'pandas>=2.2,<4', 'tqdm': 'tqdm==4.67.3'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Colab runtime with PyTorch installed.')
# Keep the tokenizer version consistent with the approved snapshot.
import importlib.metadata
if importlib.metadata.version('nltk') != '3.9.1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1'])

source_dir = str(ROOT / 'src')
sys.path.insert(0, source_dir)
for name in ('runtime_utils', 'yelp_data', 'data_runtime', 'models', 'metrics',
             'training', 'evaluation', 'review', 'reporting', 'evaluator'):
    module = sys.modules.get(name)
    if module is not None and Path(getattr(module, '__file__', '')).parent != ROOT / 'src':
        del sys.modules[name]
from data_runtime import verify_preprocessing
verify_preprocessing(ROOT)
import torch
import pandas as pd
print('Member folder:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Approved preprocessing restored; original notebook 01 is preserved.')


Mounted at /content/drive
Member folder: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary
GPU: NVIDIA A100-SXM4-40GB
Approved preprocessing restored; original notebook 01 is preserved.


## Select real errors and download annotation templates

In [2]:
from review import prepare_review, create_analysis_template
ERROR_ROWS = prepare_review(ROOT, DRIVE_BASE, CACHE_ROOT)
display(pd.DataFrame(ERROR_ROWS)[['error_id','group','source_index','true_label','predicted_label','predicted_class_confidence','distance_from_threshold','slice_memberships']])
files.download(str(ROOT / 'outputs/errors/error_annotations_template.csv'))
files.download(str(create_analysis_template(ROOT)))


,error_id,group,source_index,true_label,predicted_label,predicted_class_confidence,distance_from_threshold,slice_memberships
0,1,confident_false_positive,29330,0,1,0.999996,0.499996,short_reviews
1,2,confident_false_positive,12480,0,1,0.999910,0.499910,short_reviews
2,3,confident_false_positive,34454,0,1,0.999690,0.499690,medium_reviews;contains_negation;high_oov_rate
3,4,confident_false_positive,20008,0,1,0.999381,0.499381,medium_reviews;contains_negation
4,5,confident_false_positive,27375,0,1,0.999366,0.499366,medium_reviews;contains_negation
5,6,confident_false_negative,36001,1,0,0.999911,0.499911,short_reviews
6,7,confident_false_negative,22807,1,0,0.999901,0.499901,medium_reviews;contains_negation
7,8,confident_false_negative,3223,1,0,0.999898,0.499898,short_reviews
8,9,confident_false_negative,30793,1,0,0.999809,0.499809,medium_reviews;contains_negation
9,10,confident_false_negative,3485,1,0,0.999771,0.499771,medium_reviews;contains_negation


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Write your own analysis

Open the downloaded CSV and read all 20 full reviews. Fill ONLY error_type, explanation, and testable_fix; preserve every evidence column and all 20 rows. Each fix must be an experiment you could actually test. Open student_analysis_template.json and fill every field in your own words using the actual metrics. Compare all three own models and all six team models; discuss hardware/slice differences, limitations and future work. Save both files under their original names. The next cell waits for those two edited files. You may stop here and return later; everything is saved in Drive. No automatic interpretations or error explanations are supplied.

In [5]:
from review import import_annotations, import_student_analysis
print('Upload your completed error_annotations_template.csv AND student_analysis_template.json.')
for name in ['error_annotations_template.csv', 'student_analysis_template.json']:
    Path(name).unlink(missing_ok=True)

completed = files.upload()
needed = ['error_annotations_template.csv', 'student_analysis_template.json']
if not all(name in completed for name in needed):
    raise ValueError('Select both completed files together; preserve their filenames.')
upload_dir = ROOT / 'inputs/manual_uploads'
upload_dir.mkdir(parents=True, exist_ok=True)
for name in needed:
    (upload_dir / name).write_bytes(completed[name])
ANNOTATED_ROWS = import_annotations(ROOT, upload_dir / needed[0])
STUDENT_ANALYSIS = import_student_analysis(ROOT, upload_dir / needed[1])
print('Twenty manual annotations and student-written findings validated.')


Upload your completed error_annotations_template.csv AND student_analysis_template.json.


Saving error_annotations_template.csv to error_annotations_template.csv
Saving student_analysis_template.json to student_analysis_template.json
Twenty manual annotations and student-written findings validated.


In [6]:
from reporting import refresh_reports, export_artifacts
refresh_reports(ROOT)
print('Evidence:', export_artifacts(ROOT, '06_Manual_Review_Evidence'))
print('Download this executed notebook before opening 07.')


Evidence: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary/exports/Viraat_Task2_06_Manual_Review_Evidence.zip
Download this executed notebook before opening 07.
